In [14]:
from TableToDict import GetDFv2, GetDF, DisplayColumns
from HuRI import GetHuRI
#from ccsblib import  huri
from  Load_CCSB_Data import GetOrfMapping
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import seaborn as sns


from PandasToMySQL import MakeSummaryTable, update_column


%matplotlib inline
%config InlineBackend.figure_format='retina'

pd.options.display.max_rows = 1000
pd.options.display.max_columns = 60

In [2]:
import glob
from PlateDict import rev_wellMap, wellMap
from scipy.stats import pearsonr, spearmanr, shapiro, ttest_ind, ttest_rel
from sklearn.metrics import r2_score
from matplotlib_venn import venn2

A01
31


In [4]:
allele_collection_df = GetDFv2()
mut_d = dict()
for orf_id,rc4_mut_id,cegs_mut_id,vus_mut_id,aa_change,clinical_significance in zip(allele_collection_df.orf_id,allele_collection_df.rc4_mut_id,allele_collection_df.cegs_mut_id,allele_collection_df.vus_mut_id,allele_collection_df.aa_change,allele_collection_df.clinical_significance):
    for m_id in [rc4_mut_id,cegs_mut_id,vus_mut_id]:
        if m_id == m_id:
            mut_d[(orf_id,m_id)] = (aa_change,clinical_significance)
                                                                                                             
allele_collection_df.head()

,orf_id,nt_change,aa_change,symbol,ensg,entrez,rc4_mut_id,cegs_mut_id,vus_mut_id,rc4_plate,rc4_well,cegs_plate,cegs_well,vus_plate,vus_well,chr,chr_pos_37,chr_pos_38,cv_allele_id,hgmd_id,cv_clinical_significance,cv_phenotypes,cv_phenotype_ids,hgmd_clinical_significance,hgmd_phenotype,clinical_significance
0,2,160G>C,Val54Leu,GBA,ENSG00000177628,2629,6.0,NaN,NaN,RC4_Mut_GDEh1026,H01,None,None,None,None,1,155209824,155240033,NaN,CM960691,None,None,None,DM,Gaucher disease,Pathogenic
1,2,670C>T,Leu224Phe,GBA,ENSG00000177628,2629,73.0,NaN,NaN,RC4_Mut_GDEh1026,E01,None,None,None,None,1,155208016,155238225,NaN,CM057091,None,None,None,DM?,Gaucher disease,Pathogenic
2,2,887G>A,Arg296Gln,GBA,ENSG00000177628,2629,113.0,NaN,NaN,RC4_Mut_GDEh1026,F01,None,None,None,None,1,155207244,155237453,19367.0,CM940809,Pathogenic,"Gaucher disease;Gaucher disease, perinatal let...","MedGen:C0017205,Orphanet:ORPHA355;MedGen:C1842...",DM,Gaucher disease,Pathogenic
3,2,1448T>C,Leu483Pro,GBA,ENSG00000177628,2629,231.0,NaN,NaN,RC4_Mut_GDEh1026,G01,None,None,None,None,1,155205043,155235252,19327.0,CM870010,"Conflicting interpretations of pathogenicity, ...",Abnormality of movement;Acute neuronopathic Ga...,"Human Phenotype Ontology:HP:0100022,MedGen:C00...",DM,Gaucher disease 2,Conflicting
4,25,328C>G,Leu110Val,SLC9A3R1,ENSG00000109062,9368,252.0,200008.0,NaN,RC4_Mut_GDEh1008,F09,CegsMutGDEh1043,G10,None,None,17,72745313,74749174,20309.0,CM085716,Conflicting interpretations of pathogenicity,"Nephrolithiasis/osteoporosis, hypophosphatemic, 2","MedGen:C2676782,OMIM:612287",DM?,Nephrolithiasis/bone demineralization,Conflicting


# import experiment data

In [5]:
def calculate_NLR(xy,x_f2,y_f1):
    nlr = xy/max(x_f2,y_f1)
    if nlr < 1:
        nlr = 1
    return(np.log2(nlr))

## WT PWT data

In [7]:
res = {}
for elem in glob.glob('../20210302_yN2H_WT_PWT/data/*'):
    if 'Plate' in elem:
        
        retest_pla = int(elem.split('_')[4][-2:])
        
        df = pd.read_excel(elem, skiprows=10, skipfooter=35, engine='xlrd')
        df.rename(columns={c:i+1 for i,c in enumerate(df.columns)}, inplace=True)
        
        value_d = dict()
        for l, (i, row) in zip(['A','B','C','D','E','F','G','H'], df.iterrows()):
            for i, value in enumerate(row):
                key = (retest_pla,f'{l}{str(i+1).zfill(2)}')
                
                res[key] = value

*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: wi

In [8]:
col = ['ad_orf_id', 'ad_symbol', 'db_orf_id',
       
       'db_symbol', 'retest_pla', 'retest_pos',
       'standard_batch', 'variant_id']


query = f'select distinct { ", ".join(c for c in col)} from vus.retest where `standard_batch` in ("yN2Hwtpwt");'

yN2H_wt_retest = GetDFv2(query, col)
yN2H_wt_retest['yN2H_score'] = [res[(pla, pos)] for pla, pos in zip(yN2H_wt_retest.retest_pla, yN2H_wt_retest.retest_pos)]

empty_N1_d = dict(zip(yN2H_wt_retest[yN2H_wt_retest.variant_id == 'empty_N1'].db_orf_id, yN2H_wt_retest[yN2H_wt_retest.variant_id == 'empty_N1'].yN2H_score))
empty_N2_d = dict(zip(yN2H_wt_retest[yN2H_wt_retest.variant_id == 'empty_N2'].ad_orf_id, yN2H_wt_retest[yN2H_wt_retest.variant_id == 'empty_N2'].yN2H_score))


yN2H_wt_normalized = yN2H_wt_retest[yN2H_wt_retest.variant_id.isin(['wt','RRS','PRS','litbm'])].copy()
#yN2H_wt_normalized['pair'] = [tuple(sorted([x,y])) for x, y in zip(yN2H_wt_normalized.ad_orf_id, yN2H_wt_normalized.db_orf_id)]
yN2H_wt_normalized['variant_id'] = yN2H_wt_normalized.variant_id.apply(lambda x: f'{x}1')
yN2H_wt_normalized['db_mut_id'] = 0
yN2H_wt_normalized['aa_change'] = 'wt1'
yN2H_wt_normalized['aa_pos'] = 0
yN2H_wt_normalized['significance'] = 'wt1'

yN2H_wt_normalized['empty_N1'] = yN2H_wt_normalized.db_orf_id.map(empty_N1_d)
yN2H_wt_normalized['empty_N2'] = yN2H_wt_normalized.ad_orf_id.map(empty_N2_d)

yN2H_wt_normalized['NLR'] = [calculate_NLR(N1N2,N1F2,N2F1) for N1N2, N1F2, N2F1 in zip(yN2H_wt_normalized.yN2H_score, yN2H_wt_normalized.empty_N1, yN2H_wt_normalized.empty_N2)]

yN2H_wt_normalized.drop(columns=['retest_pla', 'retest_pos', 'standard_batch', 'empty_N1', 'empty_N2', 'yN2H_score'],inplace=True)

#yN2H_wt_normalized['NLD'] = [N1N2-N1F2-N2F1 for N1N2, N1F2, N2F1 in zip(yN2H_wt_normalized.yN2H_score, yN2H_wt_normalized.empty_N1, yN2H_wt_normalized.empty_N2)]
yN2H_wt_normalized['batch'] = 'wt_pwt'
yN2H_wt_normalized.head()

,ad_orf_id,ad_symbol,db_orf_id,db_symbol,variant_id,db_mut_id,aa_change,aa_pos,significance,NLR,batch
2,53053,CCHCR1,386,STAR,litbm1,0,wt1,0,wt1,0.0,wt_pwt
3,52859,CCDC136,54082,PRPF31,wt1,0,wt1,0,wt1,0.0,wt_pwt
8,6759,RPS27A,52920,LITAF,wt1,0,wt1,0,wt1,0.0,wt_pwt
11,70408,OXER1,5933,TFG,wt1,0,wt1,0,wt1,0.0,wt_pwt
12,14862,GOLGA2,54459,RNF135,wt1,0,wt1,0,wt1,0.0,wt_pwt


In [12]:
yN2H_wt_retest.head()

,ad_orf_id,ad_symbol,db_orf_id,db_symbol,retest_pla,retest_pos,standard_batch,variant_id,yN2H_score
0,0,None,71258,CTNNA3,1,A01,yN2Hwtpwt,empty_N1,124
1,9580,GPR161,0,None,1,B01,yN2Hwtpwt,empty_N2,206
2,53053,CCHCR1,386,STAR,1,C01,yN2Hwtpwt,litbm,96
3,52859,CCDC136,54082,PRPF31,1,D01,yN2Hwtpwt,wt,89
4,644,ITPK1,0,None,1,F01,yN2Hwtpwt,empty_N2,124


In [9]:
yN2H_wt_normalized.variant_id.value_counts()

wt1       699
RRS1       78
PRS1       60
litbm1     56
Name: variant_id, dtype: int64

In [ ]:
# yN2H_wt_d = {(ad,db):(yN2H_score,empty_N1,empty_N2,NLR,NLD) for ad,db,yN2H_score,empty_N1,empty_N2,NLR,NLD in zip(yN2H_wt_normalized.ad_orf_id,
#                                                                                                                   yN2H_wt_normalized.db_orf_id,
#                                                                                                                   yN2H_wt_normalized.yN2H_score,
#                                                                                                                   yN2H_wt_normalized.empty_N1,
#                                                                                                                   yN2H_wt_normalized.empty_N2,
#                                                                                                                   yN2H_wt_normalized.NLR)}

## yN2H allele pairwise test

In [10]:
res = {}
for f in glob.glob('20210329_VUS_yN2Hapwt/*.xls'):
    

    retest_pla = int(f.split('_')[3].split('Plate')[-1])
    if  retest_pla < 38:
        replicate = 1
    else:
        retest_pla -= 37
        replicate = 2
        
    df = pd.read_excel(f, skiprows=10, skipfooter=35, engine='xlrd')
    df.rename(columns={c:i+1 for i,c in enumerate(df.columns)}, inplace=True)

    value_d = dict()
    for l, (i, row) in zip(['A','B','C','D','E','F','G','H'], df.iterrows()):
        for i, value in enumerate(row):
            key = (retest_pla,replicate,f'{l}{str(i+1).zfill(2)}')
            res[key] = value

*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: wi

In [11]:
col = ['ad_orf_id', 'ad_symbol', 'db_orf_id',
       'db_symbol', 'db_mut_id', 'retest_pla', 'retest_pos',
       'standard_batch', 'variant_id']

query = f'select distinct { ", ".join(c for c in col)} from vus.retest where `standard_batch` in ("yN2Hapwt");'

yN2H_retest = GetDFv2(query, col)
yN2H_retest['N1N2r1'] = [res[(pla, 1, pos)] for pla, pos in zip(yN2H_retest.retest_pla, yN2H_retest.retest_pos)]
yN2H_retest['N1N2r2'] = [res[(pla, 2, pos)] for pla, pos in zip(yN2H_retest.retest_pla, yN2H_retest.retest_pos)]

yN2H_retest.head()

,ad_orf_id,ad_symbol,db_orf_id,db_symbol,db_mut_id,retest_pla,retest_pos,standard_batch,variant_id,N1N2r1,N1N2r2
0,71963,MBD3L1,2713,MLH1,981.0,4,H05,yN2Hapwt,allele,96,82
1,11452,WASHC1,4723,TPM1,201730.0,4,G05,yN2Hapwt,allele,336,103
2,361,GRAMD2B,3438,TMEM43,0.0,4,F05,yN2Hapwt,wt,6558,2033
3,71892,TH,71892,TH,4005.0,4,E05,yN2Hapwt,allele,1092,254
4,2500,ZNF212,2713,MLH1,0.0,4,D05,yN2Hapwt,wt,124,82


In [ ]:
empty_well_df = pd.read_excel('20210328_yN2H_empty_wells.xlsx',sheet_name='Sheet2')
empty_well_df['replicate'] = empty_well_df.Plates.apply(lambda x: int(x.split('_')[2][-1]))
empty_well_df['plate'] = empty_well_df.Plates.apply(lambda x: int(x.split('_')[-1].replace('Plate','')))
empty_well_df['empty_wells'] = empty_well_df['Empty wells'].apply(lambda x: {f"{e.replace(' ','')[0]}{e.replace(' ','')[1:].zfill(2)}" for e in x.split(',')})
#empty_well_d = {(plate,rep):wells for plate, rep, wells in zip(empty_well_df.plate, empty_well_df.replicate, empty_well_df.empty_wells)}       
empty_well_set =  {(plate, rep, well) for plate, rep, wells in zip(empty_well_df.plate, empty_well_df.replicate, empty_well_df.empty_wells) for well in wells }

query = '''select orf_id, mut_id from vus.N2H_validation_nodes where collection = "rc4" '''
col = ['db_orf_id','db_mut_id']
rc4_mut_ids = GetDFv2(query,col)
rc4_mut_ids = set(zip(rc4_mut_ids.db_orf_id, rc4_mut_ids.db_mut_id))

yN2H_retest['empty_well_rep1'] = [(db_orf_id, db_mut_id) in rc4_mut_ids or (plate,1,retest_pos) in empty_well_set for db_orf_id, db_mut_id, plate, retest_pos in zip(yN2H_retest.db_orf_id, yN2H_retest.db_mut_id, yN2H_retest.retest_pla, yN2H_retest.retest_pos)]
yN2H_retest['empty_well_rep2'] = [(db_orf_id, db_mut_id) in rc4_mut_ids or (plate,2,retest_pos) in empty_well_set for db_orf_id, db_mut_id, plate, retest_pos in zip(yN2H_retest.db_orf_id, yN2H_retest.db_mut_id, yN2H_retest.retest_pla, yN2H_retest.retest_pos)]

yN2H_retest['N1N2r1'] = [score if not empty else np.nan for score, empty in zip(yN2H_retest.N1N2r1, yN2H_retest.empty_well_rep1)]
yN2H_retest['N1N2r2'] = [score if not empty else np.nan for score, empty in zip(yN2H_retest.N1N2r2, yN2H_retest.empty_well_rep2)]

yN2H_retest.head()

In [ ]:
empty_N1_rep1_d = {(db,db_mut_id) if db_mut_id == db_mut_id else (db,999999):score for db, db_mut_id, score in zip(yN2H_retest[yN2H_retest.variant_id == 'empty_N1'].db_orf_id, 
                                                                                                                   yN2H_retest[yN2H_retest.variant_id == 'empty_N1'].db_mut_id, 
                                                                                                                   yN2H_retest[yN2H_retest.variant_id == 'empty_N1'].N1N2r1)}
empty_N2_rep1_d = dict(zip(yN2H_retest[yN2H_retest.variant_id == 'empty_N2'].ad_orf_id, yN2H_retest[yN2H_retest.variant_id == 'empty_N2'].N1N2r1))

empty_N1_rep2_d = {(db,db_mut_id) if db_mut_id == db_mut_id else (db,999999):score for db, db_mut_id, score in zip(yN2H_retest[yN2H_retest.variant_id == 'empty_N1'].db_orf_id, 
                                                                                                                   yN2H_retest[yN2H_retest.variant_id == 'empty_N1'].db_mut_id, 
                                                                                                                   yN2H_retest[yN2H_retest.variant_id == 'empty_N1'].N1N2r2)}
empty_N2_rep2_d = dict(zip(yN2H_retest[yN2H_retest.variant_id == 'empty_N2'].ad_orf_id, yN2H_retest[yN2H_retest.variant_id == 'empty_N2'].N1N2r2))
 

yN2H_normalized = yN2H_retest[yN2H_retest.variant_id.isin(['wt', 'allele', 'RRS','PRS'])].copy()

yN2H_normalized['EN1_rep1'] = [empty_N1_rep1_d[(db_orf_id, db_mut_id)] if db_mut_id == db_mut_id else empty_N1_rep1_d[(db_orf_id, 999999)] for db_orf_id,db_mut_id in zip(yN2H_normalized.db_orf_id, yN2H_normalized.db_mut_id)]
yN2H_normalized['EN2_rep1'] = yN2H_normalized.ad_orf_id.map(empty_N2_rep1_d)

yN2H_normalized['EN1_rep2'] = [empty_N1_rep2_d[(db_orf_id, db_mut_id)] if db_mut_id == db_mut_id else empty_N1_rep2_d[(db_orf_id, 999999)] for db_orf_id,db_mut_id in zip(yN2H_normalized.db_orf_id, yN2H_normalized.db_mut_id)]
yN2H_normalized['EN2_rep2'] = yN2H_normalized.ad_orf_id.map(empty_N2_rep2_d)

yN2H_normalized['NLR1'] = [calculate_NLR(N1N2,N1F2,N2F1) for N1N2, N1F2, N2F1 in zip(yN2H_normalized.N1N2r1, yN2H_normalized.EN1_rep1, yN2H_normalized.EN2_rep1)]
yN2H_normalized['NLR2'] = [calculate_NLR(N1N2,N1F2,N2F1) for N1N2, N1F2, N2F1 in zip(yN2H_normalized.N1N2r2, yN2H_normalized.EN1_rep2, yN2H_normalized.EN2_rep2)]

min_nld_r1 = min([(N1N2-N1F2-N2F1) for N1N2, N1F2, N2F1 in zip(yN2H_normalized.N1N2r1, yN2H_normalized.EN1_rep1, yN2H_normalized.EN2_rep1)])
min_nld_r2 = min([(N1N2-N1F2-N2F1) for N1N2, N1F2, N2F1 in zip(yN2H_normalized.N1N2r2, yN2H_normalized.EN1_rep2, yN2H_normalized.EN2_rep2)])

yN2H_normalized['NLD1'] = [np.log10((N1N2-N1F2-N2F1)+abs(min_nld_r1)+1) -np.log10(abs(min_nld_r1)) for N1N2, N1F2, N2F1 in zip(yN2H_normalized.N1N2r1, yN2H_normalized.EN1_rep1, yN2H_normalized.EN2_rep1)]
yN2H_normalized['NLD2'] = [np.log10((N1N2-N1F2-N2F1)+abs(min_nld_r2)+1) -np.log10(abs(min_nld_r2)) for N1N2, N1F2, N2F1 in zip(yN2H_normalized.N1N2r2, yN2H_normalized.EN1_rep2, yN2H_normalized.EN2_rep2)]

yN2H_normalized['NLR1'] = [v if e == False else np.nan for v, e in zip(yN2H_normalized.NLR1, yN2H_normalized.empty_well_rep1)]
yN2H_normalized['NLR2'] = [v if e == False else np.nan for v, e in zip(yN2H_normalized.NLR2, yN2H_normalized.empty_well_rep2)]
yN2H_normalized['NLD1'] = [v if e == False else np.nan for v, e in zip(yN2H_normalized.NLD1, yN2H_normalized.empty_well_rep1)]
yN2H_normalized['NLD2'] = [v if e == False else np.nan for v, e in zip(yN2H_normalized.NLD2, yN2H_normalized.empty_well_rep2)]

yN2H_normalized['NLR'] = [np.mean([v for v in [v1, v2] if v == v]) if len([v for v in [v1, v2] if v == v]) > 0 else np.nan for v1, v2 in zip(yN2H_normalized.NLR1, yN2H_normalized.NLR2)]
yN2H_normalized['NLD'] = [np.mean([v for v in [v1, v2] if v == v]) if len([v for v in [v1, v2] if v == v]) > 0 else np.nan for v1, v2 in zip(yN2H_normalized.NLD1, yN2H_normalized.NLD2)]

yN2H_normalized['pair'] = [tuple(sorted([ad,db])) for ad, db in zip(yN2H_normalized.ad_orf_id, yN2H_normalized.db_orf_id)]

yN2H_normalized.head()

In [ ]:
yN2H_normalized[yN2H_normalized.variant_id == 'RRS'].NLR1.max()

In [ ]:
yN2H_normalized[yN2H_normalized.variant_id == 'RRS'].NLR2.max()

In [ ]:
yN2H_normalized[yN2H_normalized.variant_id == 'RRS'].NLR.max()

In [ ]:
yN2H_normalized.NLR.plot.hist(bins=100)
plt.ylabel('Count')
plt.xlabel('NLR (log2)')

In [ ]:
# plt.figure()
# yN2H_normalized.NLD.plot.hist()

## qY2H data

In [ ]:
col  = ['ad_orf_id', 'ad_symbol', 'db_orf_id',
       'db_symbol', 'db_mut_id', 'standard_batch', 'retest_pla', 'retest_pos', 'LW',
       'LWH1', 'LWH10', 'LWH25', 'LWA', 'LWAH1']

query = f'select {", ".join(c for c in col)} from vus.retest where standard_batch in ("VUSAPWT1B1", "VUSAPWT1B2", "VUSAPWT2B1", "VUSAPWT6B1");'

VUSAPWT_1B2_2B1_6B1 = GetDFv2(query, col)

df = VUSAPWT_1B2_2B1_6B1[(~VUSAPWT_1B2_2B1_6B1.db_orf_id.isnull())&(VUSAPWT_1B2_2B1_6B1.ad_orf_id == 0)&(VUSAPWT_1B2_2B1_6B1.LW >= 1)]

EAD_d = {(db_orf_id,db_mut_id,sb):{'LWH1':{}, 'LWH10':{}, 'LWH25':{},'LWA':{}, 'LWAH1':{}} for db_orf_id,db_mut_id,sb in  zip(df.db_orf_id, df.db_mut_id, df.standard_batch)}

for db_orf_id, db_mut_id, LW, LWH1, LWH10, LWH25, LWA, LWAH1,sb in zip(df.db_orf_id, df.db_mut_id, df.LW,df.LWH1,df.LWH10,df.LWH25, df.LWA, df.LWAH1, df.standard_batch):
    for media, score in zip(['LWH1', 'LWH10', 'LWH25', 'LWA', 'LWAH1'],[LWH1, LWH10, LWH25, LWA, LWAH1]):
        EAD_d[(db_orf_id,db_mut_id,sb)][media] = score


def get_final_score(score, ead_score, LW_score):
    res = np.nan
    
    if score == score and ead_score == ead_score and LW_score == LW_score:
        if LW_score > 0:
            if score == 0:
                res = score
            elif ead_score == 0:
                res = score
            else:
                res = score - ead_score if score - ead_score >= 2 else np.nan
            
    return res
        
        
for media in ['LWH1', 'LWH10', 'LWH25', 'LWA', 'LWAH1']:
    VUSAPWT_1B2_2B1_6B1[f'EAD_{media}'] = [EAD_d[(db_orf_id, db_mut_id,sb)][media] if (db_orf_id, db_mut_id,sb) in EAD_d else np.nan for db_orf_id, db_mut_id, sb in zip(VUSAPWT_1B2_2B1_6B1.db_orf_id, VUSAPWT_1B2_2B1_6B1.db_mut_id, VUSAPWT_1B2_2B1_6B1.standard_batch)]
    VUSAPWT_1B2_2B1_6B1[f'{media}_f'] = [get_final_score(score, ead_score,LW_score) for score, ead_score, LW_score in zip(VUSAPWT_1B2_2B1_6B1[media], VUSAPWT_1B2_2B1_6B1[f'EAD_{media}'], VUSAPWT_1B2_2B1_6B1.LW)]
    
VUSAPWT_1B2_2B1_6B1['sum_score'] = [sum([elem for elem in [a,b,c,d,e] if elem == elem ]) if len([elem for elem in [a,b,c,d,e] if elem == elem ]) != 0 else np.nan for a,b,c,d,e in zip(VUSAPWT_1B2_2B1_6B1.LWH1_f , VUSAPWT_1B2_2B1_6B1.LWH10_f , VUSAPWT_1B2_2B1_6B1.LWH25_f , VUSAPWT_1B2_2B1_6B1.LWA_f,  VUSAPWT_1B2_2B1_6B1.LWAH1_f)]

#VUSAPWT_1B2_2B1_6B1['prod_score'] = [np.prod([elem for elem in [a,b,c,d,e] if elem == elem  and elem > 0]) if len([elem for elem in [a,b,c,d,e] if elem == elem ]) != 0 else np.nan for a,b,c,d,e in zip(VUSAPWT_1B2_2B1_6B1.LWH1_f , VUSAPWT_1B2_2B1_6B1.LWH10_f , VUSAPWT_1B2_2B1_6B1.LWH25_f , VUSAPWT_1B2_2B1_6B1.LWA_f,  VUSAPWT_1B2_2B1_6B1.LWAH1_f)]

VUSAPWT_1B2_2B1_6B1 = VUSAPWT_1B2_2B1_6B1[(VUSAPWT_1B2_2B1_6B1.db_mut_id >= 0)&(VUSAPWT_1B2_2B1_6B1.ad_orf_id != 0)].copy()

qY2H_d = {(db_orf_id, db_mut_id, ad_orf_id):score for db_orf_id,db_mut_id,ad_orf_id,score in zip(VUSAPWT_1B2_2B1_6B1.db_orf_id,
                                                                                                 VUSAPWT_1B2_2B1_6B1.db_mut_id,
                                                                                                 VUSAPWT_1B2_2B1_6B1.ad_orf_id,
                                                                                                 VUSAPWT_1B2_2B1_6B1.sum_score)}

VUSAPWT_1B2_2B1_6B1.head()

In [ ]:
yN2H_normalized['qY2H'] = [qY2H_d[(db,m_id,ad)] if (db,m_id,ad) in qY2H_d else np.nan for db,m_id,ad in zip(yN2H_normalized.db_orf_id, yN2H_normalized.db_mut_id, yN2H_normalized.ad_orf_id)]

yN2H_normalized.head()

## Update raw data database

In [ ]:
col = ['retest_pla', 'retest_pos', 'standard_batch','xp1', 'xp2']

condition_columns = ['retest_pla', 'retest_pos', 'standard_batch'] 

condition_col_values_d = {(retest_pla, retest_pos, standard_batch):{'N2H_raw_r01':xp1, 'N2H_raw_r02':xp2} for retest_pla, retest_pos,standard_batch,xp1, xp2 in zip(retest_df.retest_pla,
                                                                                                                                                   retest_df.retest_pos,
                                                                                                                                                   retest_df.standard_batch,
                                                                                                                                                   retest_df.xp1,
                                                                                                                                                    retest_df.xp2)}
#update_column('vus', 'retest', condition_columns, condition_col_values_d)
                                                                            

### WT  pwt

In [21]:
col = ['retest_pla', 'retest_pos', 'standard_batch','xp1', 'xp2']

condition_columns = ['retest_pla', 'retest_pos', 'standard_batch'] 

condition_col_values_d = {(retest_pla, retest_pos, standard_batch):{'N2H_raw_r01':yN2H_score} for retest_pla, retest_pos,standard_batch,yN2H_score  in zip(yN2H_wt_retest.retest_pla,
                                                                                                                                                           yN2H_wt_retest.retest_pos,
                                                                                                                                                           yN2H_wt_retest.standard_batch,
                                                                                                                                                           yN2H_wt_retest.yN2H_score)}

#update_column('vus', 'retest', condition_columns, condition_col_values_d)
                          

### Allele pwt

In [25]:
col = ['retest_pla', 'retest_pos', 'standard_batch','xp1', 'xp2']

condition_columns = ['retest_pla', 'retest_pos', 'standard_batch'] 

condition_col_values_d = {(retest_pla, retest_pos, standard_batch):{'N2H_raw_r01':N1N2r1, 'N2H_raw_r02':N1N2r2} for retest_pla, retest_pos,standard_batch,N1N2r1,N1N2r2  in zip(yN2H_retest.retest_pla,
                                                                                                                                                                                   yN2H_retest.retest_pos,
                                                                                                                                                                                   yN2H_retest.standard_batch,
                                                                                                                                                                                   yN2H_retest.N1N2r1,
                                                                                                                                                                                   yN2H_retest.N1N2r2
                                                                                                                                                                                   )}

#update_column('vus', 'retest', condition_columns, condition_col_values_d)


In [15]:
yN2H_retest.head()

,ad_orf_id,ad_symbol,db_orf_id,db_symbol,db_mut_id,retest_pla,retest_pos,standard_batch,variant_id,N1N2r1,N1N2r2
0,71963,MBD3L1,2713,MLH1,981.0,4,H05,yN2Hapwt,allele,96,82
1,11452,WASHC1,4723,TPM1,201730.0,4,G05,yN2Hapwt,allele,336,103
2,361,GRAMD2B,3438,TMEM43,0.0,4,F05,yN2Hapwt,wt,6558,2033
3,71892,TH,71892,TH,4005.0,4,E05,yN2Hapwt,allele,1092,254
4,2500,ZNF212,2713,MLH1,0.0,4,D05,yN2Hapwt,wt,124,82


# Analysis

##  Noise and intensity

In [ ]:
noise_df = yN2H_normalized[(~yN2H_normalized.NLR.isnull())&(yN2H_normalized.empty_well_rep1==False)&(yN2H_normalized.empty_well_rep2==False)].copy()
#mut_d[(orf_id,m_id)] = (aa_change,clinical_significance)
noise_df['aa_change'] =  [mut_d[o,m][0] if (o,m) in mut_d else 'wt' if v == 'wt' else np.nan for o,m,v in zip(noise_df.db_orf_id, noise_df.db_mut_id,noise_df.variant_id)]
noise_df['aa_pos'] = noise_df.aa_change.apply(lambda x: 0 if x == 'wt' else int(x[3:-3]) if x == x else np.nan)
noise_df['significance'] =  [mut_d[o,m][1] if (o,m) in mut_d else 'wt' if v == 'wt' else np.nan for o,m,v in zip(noise_df.db_orf_id, noise_df.db_mut_id,noise_df.variant_id)]

col = ['ad_orf_id', 'ad_symbol', 'db_orf_id', 'db_symbol', 'db_mut_id','aa_change', 'aa_pos','significance',
       'variant_id','qY2H',
       'N1N2r1', 'N1N2r2', 'EN1_rep1', 'EN2_rep1', 'EN1_rep2', 'EN2_rep2',
       'NLR1', 'NLR2', 'NLD1', 'NLD2', 'NLR', 'NLD']


noise_df =  noise_df[col].copy()
print(len(noise_df))
noise_df = pd.concat([yN2H_wt_normalized,noise_df])

df = noise_df[noise_df.variant_id == 'wt']
wt_d = {(ad,db):(NLR,NLD,qY2H,abs(NLR1-NLR2) ,abs(NLD1-NLD2))for ad,db,NLR,NLD,qY2H,NLR1,NLD1,NLR2,NLD2 in zip(df.ad_orf_id, df.db_orf_id, df.NLR, df.NLD, df.qY2H, df.NLR1, df.NLD1, df.NLR2, df.NLD2)}

noise_df['WT_NLR'] =  [wt_d[ad,db][0] if (ad,db) in wt_d else np.nan for ad, db in zip(noise_df.ad_orf_id, noise_df.db_orf_id)]
#noise_df['WT_NLD'] =  [wt_d[ad,db][1] if (ad,db) in wt_d else np.nan for ad, db in zip(noise_df.ad_orf_id, noise_df.db_orf_id)]
#noise_df['WT_NLRd'] =  [wt_d[ad,db][3] if (ad,db) in wt_d else np.nan for ad, db in zip(noise_df.ad_orf_id, noise_df.db_orf_id)]
#noise_df['WT_NLDd'] =  [wt_d[ad,db][4] if (ad,db) in wt_d else np.nan for ad, db in zip(noise_df.ad_orf_id, noise_df.db_orf_id)]
noise_df['WT_qY2H'] =  [wt_d[ad,db][2] if (ad,db) in wt_d else np.nan for ad, db in zip(noise_df.ad_orf_id, noise_df.db_orf_id)]

#noise_df['NLRd'] = [NLR1/NLR2 if NLR2 != 0 else np.nan for NLR1,NLR2 in zip(noise_df.NLR1, noise_df.NLR2)]
#noise_df['NLDd'] = [NLD1/NLD2 if NLD1 != 0 and NLD2 != 0 else np.nan for NLD1,NLD2 in zip(noise_df.NLD1, noise_df.NLD2)]
#noise_df['NLDd'] = [NLD1/NLD2 if NLD2 != 0 else np.nan for NLD1,NLD2 in zip(noise_df.NLD1, noise_df.NLD2)]

#noise_df['NLRp'] = [WT_NLR/NLR if NLR != 0 else np.nan for NLR,WT_NLR in zip(noise_df.NLR, noise_df.WT_NLR)]
#noise_df['NLDp'] = [np.log2(abs(WT_NLD/NLD)) if NLD != 0 else np.nan for NLD,WT_NLD in zip(noise_df.NLD, noise_df.WT_NLD)]

#noise_df['zNLRd'] = noise_df.NLRd.apply(lambda x: ((x-noise_df.NLRd.mean())/np.std(noise_df.NLRd)))
#noise_df['zNLRp'] = noise_df.NLRp.apply(lambda x: ((x-noise_df.NLRd.mean())/np.std(noise_df.NLRd)))

#noise_df['zNLDd'] = noise_df.NLRd.apply(lambda x: ((x-noise_df.NLRd.mean())/np.std(noise_df.NLRd)))
#noise_df['zNLDp'] = noise_df.NLRp.apply(lambda x: ((x-noise_df.NLRd.mean())/np.std(noise_df.NLRd)))

noise_df.drop(columns=['NLR1', 'NLR2', 'NLD1', 'NLD2', 'NLD'],inplace=True)

noise_df['qY2Hp'] = [np.nan if WT_qY2H == 0 else (WT_qY2H+1)/(qY2H+1) for qY2H,WT_qY2H in zip(noise_df.qY2H, noise_df.WT_qY2H)]
noise_df['qY2Hp'] = [np.nan if WT_qY2H == 0 else np.log2((WT_qY2H+1)/(qY2H+1)) for qY2H,WT_qY2H in zip(noise_df.qY2H, noise_df.WT_qY2H)]

rrs_mean = noise_df[noise_df.variant_id == 'RRS'].NLR.mean()
rrs_std = np.std(noise_df[noise_df.variant_id == 'RRS'].NLR)

noise_df['Zwt'] = noise_df.WT_NLR.apply(lambda x: (x-rrs_mean)/rrs_std)
noise_df['Zmt'] = noise_df.NLR.apply(lambda x: (x-rrs_mean)/rrs_std)

noise_df['pert'] = [wt-mt for wt,mt in zip(noise_df.WT_NLR, noise_df.NLR)]
noise_df['zpert'] = noise_df.pert.apply(lambda x: (x-rrs_mean)/rrs_std)
noise_df['perturbation_category'] = noise_df.zpert.apply(lambda x: 'increased NLR' if x <= -2 else 'decreased NLR' if x >= 2 else 'WT like')
noise_df['batch'] = 'apwt'

print(len(noise_df))
noise_df = noise_df[((~noise_df.WT_NLR.isnull())&(noise_df.aa_change == 'wt1'))|(noise_df.variant_id.isin(['allele', 'wt', 'RRS', 'PRS']))].copy()
print(len(noise_df))

noise_df.head()

In [ ]:
noise_df.variant_id.value_counts()

In [ ]:
rrs_mean

In [ ]:
sns.histplot(data=noise_df[noise_df.variant_id.isin(['PRS','RRS'])], x="NLR", kde=True, hue='variant_id')

In [ ]:
sns.histplot(data=noise_df[noise_df.variant_id.isin(['PRS','RRS'])], x="Zmt", kde=True, hue='variant_id')
plt.xlabel('Z-score')

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)].sort_values(by='perturbation_category').perturbation_category.value_counts().plot.pie()
plt.title('Alleles');
plt.ylabel('');
plt.show()

In [ ]:
noise_df.pert.plot.hist()

In [ ]:
sns.histplot(data=noise_df[(noise_df.variant_id.isin(['allele']))&(noise_df.Zwt >= 2)], x="pert", kde=True)
plt.xlabel('NLR perturbation');

In [ ]:
sns.histplot(data=noise_df[(noise_df.variant_id.isin(['allele']))&(noise_df.Zwt >= 2)], x="zpert", kde=True)
plt.xlabel('NLR Z-score perturbation');

In [ ]:
sns.histplot(data=noise_df[(noise_df.variant_id.isin(['allele']))&(noise_df.Zwt <= 2)], x="zpert", kde=True)
plt.xlabel('NLR Z-score perturbation');

In [ ]:
sns.histplot(data=noise_df[(noise_df.variant_id.isin(['wt1']))&(noise_df.Zwt >= 2)], x="pert", kde=True)
plt.xlabel('NLR perturbation');

In [ ]:
sns.histplot(data=noise_df[(noise_df.variant_id.isin(['wt1']))&(noise_df.Zwt >= 2)], x="zpert", kde=True)
plt.xlabel('NLR Z-score perturbation');

In [ ]:
noise_df[(noise_df.variant_id == 'wt1')&(noise_df.Zwt >= 2)].sort_values(by='perturbation_category').perturbation_category.value_counts().plot.pie()
plt.title('repeated WT PPIs');
plt.ylabel('');
plt.show()

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)&(noise_df.significance == 'Pathogenic')].perturbation_category.value_counts().plot.pie()
plt.title('Pathogenic variants');
plt.ylabel('');
plt.show()

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)&(noise_df.significance == 'Pathogenic')].perturbation_category.value_counts(normalize=True)

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)].significance.value_counts()

In [ ]:
col  = [ 'db_orf_id', 'db_symbol', 'variant_id',
       'db_mut_id', 'aa_change', 'aa_pos', 'significance']

noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)][col].drop_duplicates().significance.value_counts()

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)].significance.value_counts().plot.pie()

In [ ]:
noise_df[noise_df.db_symbol == 'STXBP1']

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt <= 2)].perturbation_category.value_counts().plot.pie()

In [ ]:
col  = ['ad_symbol', 'db_symbol', 'aa_change', 'significance', 'NLR', 
       'qY2H', 'WT_NLR', 'WT_qY2H', 'Zwt', 'Zmt', 'pert', 'zpert',
       'perturbation_category']

df  = noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt <= 2)&(noise_df.perturbation_category == 'increased NLR')][col].sort_values(by=['db_symbol','aa_change']).copy()

df.rename(columns={'ad_symbol':'N1_symbol', 'db_symbol':'N2_symbol'},inplace=True)
df

In [ ]:
rrs_mean

In [ ]:
len(noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)&(noise_df.significance == 'Pathogenic')])

In [ ]:
len(noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)&(noise_df.significance == 'VUS')])

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)&(noise_df.significance == 'VUS')].perturbation_category.value_counts()

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)&(noise_df.significance == 'VUS')].perturbation_category.value_counts().plot.pie(colors = ['darkorange','steelblue'])
plt.title('VUS');
plt.ylabel('');
plt.show()

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)&(noise_df.significance == 'Benign')].perturbation_category.value_counts().plot.pie()
plt.title('Benign variants');
plt.ylabel('');
plt.show()

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt >= 2)].zpert.plot.hist()

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt <= 2)&(noise_df.zpert <= -2)]

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.Zwt <= 2)&(noise_df.db_symbol == 'PMP22')].sort_values(by=['aa_change','ad_symbol'])

In [ ]:
noise_df[noise_df.variant_id == 'RRS'].Zmt.plot.hist()

In [ ]:
noise_df[(noise_df.Zwt >= 2)&(noise_df.variant_id == 'allele')].Zmt.plot.hist()

In [ ]:
noise_df[(noise_df.Zwt < 2)&(noise_df.variant_id == 'allele')].Zmt.plot.hist()

## perturbation relative to RRS detection

In [ ]:
noise_df.variant_id.value_counts()

In [ ]:
noise_df[noise_df.variant_id == 'RRS'].NLR1.plot.hist(bins=15)

In [ ]:
col = ['ad_orf_id', 'ad_symbol', 'db_orf_id', 'db_symbol', 'db_mut_id',
       'aa_change', 'aa_pos', 'significance', 'variant_id', 'qY2H', 'NLR', 'WT_NLR', 'WT_qY2H',
       'qY2Hp']

pert_df = noise_df[col].copy()

pert_df['WT_detected'] = pert_df.WT_NLR.apply(lambda x: x > pert_df[pert_df.variant_id == 'RRS'].NLR.max() and  x > 0)

rrs_mean = pert_df[pert_df.variant_id == 'RRS'].NLR.mean()
rrs_std = np.std(pert_df[pert_df.variant_id == 'RRS'].NLR)

pert_df['Zwt'] = pert_df.WT_NLR.apply(lambda x: (x-rrs_mean)/rrs_std)
pert_df['Zmt'] = pert_df.NLR.apply(lambda x: (x-rrs_mean)/rrs_std)
pert_df['perturbed'] = pert_df.NLR < 0

pert_df.head()

In [ ]:
pert_df[pert_df.WT_detected == True].fold_change.plot.hist()

In [ ]:
pert_df[(pert_df.WT_detected == True)&(pert_df.fold_change < 0)]

In [ ]:
pert_df.fold_change.plot.hist()

In [ ]:
np.log2(13)

In [ ]:
pert_df[(pert_df.variant_id == 'allele')&(pert_df.WT_detected)&(pert_df.WT_NLR > 6)]

In [ ]:
pert_df[pert_df.Zwt > 2]

In [ ]:
np.std(pert_df[pert_df.variant_id == 'RRS'].NLR)

In [ ]:
pert_df[pert_df.variant_id == 'RRS'].NLR.mean()

In [ ]:
noise_df.head()

In [ ]:
len(noise_df[(noise_df.WT1_detected > 0)&(noise_df.NLR1_detected == False)&(noise_df.variant_id  ==  'allele')])

In [ ]:
len(noise_df[(noise_df.WT1_detected > 0)&(noise_df.NLR1_detected == True)&(noise_df.variant_id  ==  'allele')])

In [ ]:
len(noise_df[(noise_df.WT1_detected > 0)&(noise_df.NLR1_detected == False)&(noise_df.variant_id  ==  'allele')])


In [ ]:
df2.NLR1_detected.value_counts()

In [ ]:
df1.NLR1_detected.value_counts()

## qY2H and yN2H integration / validation

In [ ]:
x = [nlr for i,nlr in enumerate(noise_df[(noise_df.variant_id == 'wt')&(noise_df.WT_qY2H > 10)&(noise_df.WT_NLR > 1)].sort_values(by='NLR').NLR)]
y = [(((noise_df.variant_id == 'wt')&(noise_df.WT_qY2H > 10)&(noise_df.WT_NLR > 1)).sum() - i )/ ((noise_df.variant_id == 'wt')&(noise_df.WT_qY2H > 10)&(noise_df.WT_NLR > 1)).sum() for i,nlr in enumerate(noise_df[(noise_df.variant_id == 'wt')&(noise_df.WT_qY2H > 10)&(noise_df.WT_NLR > 1)].sort_values(by='NLR').NLR)]
plt.plot(x,y, color='m',label='WT')

x = [nlr for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6))].sort_values(by='NLR').NLR.apply(np.log2))]+[25]
y = [(((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6)).sum() - i )/ ((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6)).sum() for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6))].sort_values(by='NLR').NLR.apply(np.log2))]+[0]
plt.plot(x,y, color='y',label='Y2H not perturbed')

x = [nlr for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H == 0))].sort_values(by='NLR').NLR.apply(np.log2))]+[25]
y = [(((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H == 0)).sum() - i )/ ((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H == 0)).sum() for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H == 0))].sort_values(by='NLR').NLR.apply(np.log2))]+[0]
plt.plot(x,y, color='g',label='Y2H perturbed')

x = [nlr for i,nlr in enumerate(noise_df[noise_df.variant_id == 'PRS'].sort_values(by='NLR').NLR.apply(np.log2))]
y = [((noise_df.variant_id == 'PRS').sum() - i )/ (noise_df.variant_id == 'PRS').sum() for i,nlr in enumerate(noise_df[noise_df.variant_id == 'PRS'].sort_values(by='NLR').NLR)]
plt.plot(x,y, color='b',label='PRS')

x = [nlr for i,nlr in enumerate(noise_df[noise_df.variant_id == 'RRS'].sort_values(by='NLR').NLR)]+[25]
y = [((noise_df.variant_id == 'RRS').sum() - i )/ (noise_df.variant_id == 'RRS').sum() for i,nlr in enumerate(noise_df[noise_df.variant_id == 'RRS'].sort_values(by='NLR').NLR)]+[0]
plt.plot(x,y, color='r',label='RRS')
plt.legend()
plt.xlabel('NLR (log2)')
plt.ylabel('Fraction positive')
plt.xlim(0.6,8)
plt.ylim(-0.05,0.8)

In [ ]:
x = [-1]+[nlr for i,nlr in enumerate(noise_df[(noise_df.variant_id == 'wt')&(noise_df.WT_qY2H > 10)&(noise_df.WT_NLR > 1)].sort_values(by='NLR').NLR)]
y = [1]+[(((noise_df.variant_id == 'wt')&(noise_df.WT_qY2H > 10)&(noise_df.WT_NLR > 1)).sum() - i )/ ((noise_df.variant_id == 'wt')&(noise_df.WT_qY2H > 10)&(noise_df.WT_NLR > 1)).sum() for i,nlr in enumerate(noise_df[(noise_df.variant_id == 'wt')&(noise_df.WT_qY2H > 10)&(noise_df.WT_NLR > 1)].sort_values(by='NLR').NLR)]
plt.plot(x,y, color='m',label='WT')

x = [nlr for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6))].sort_values(by='NLR').NLR)]+[25]
y = [(((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6)).sum() - i )/ ((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6)).sum() for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6))].sort_values(by='NLR').NLR)]+[0]
plt.plot(x,y, color='y',label='Y2H not perturbed')


x = [nlr for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H <= 6)&(noise_df.qY2H > 1))].sort_values(by='NLR').NLR)]+[25]
y = [(((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H <= 6)&(noise_df.qY2H > 1)).sum() - i )/ ((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H <= 6)&(noise_df.qY2H > 1)).sum() for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H <= 6)&(noise_df.qY2H > 1))].sort_values(by='NLR').NLR)]+[0]
plt.plot(x,y, color='r',label='Y2H perturbed')

#x = [nlr for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6))].sort_values(by='NLR').NLR)]+[25]
#y = [(((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6)).sum() - i )/ ((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6)).sum() for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H > 6))].sort_values(by='NLR').NLR)]+[0]
#plt.plot(x,y, color='y',label='Y2H not perturbed')


x = [nlr for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H == 0))].sort_values(by='NLR').NLR)]+[25]
y = [(((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H == 0)).sum() - i )/ ((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H == 0)).sum() for i,nlr in enumerate(noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H == 0))].sort_values(by='NLR').NLR)]+[0]
plt.plot(x,y, color='g',label='Y2H gone')


#x = [nlr for i,nlr in enumerate(noise_df[noise_df.variant_id == 'RRS'].sort_values(by='NLR').NLR)]+[25]
#y = [((noise_df.variant_id == 'RRS').sum() - i )/ (noise_df.variant_id == 'RRS').sum() for i,nlr in enumerate(noise_df[noise_df.variant_id == 'RRS'].sort_values(by='NLR').NLR)]+[0]
#plt.plot(x,y, color='r',label='RRS')


plt.legend()
plt.xlabel('NLR (log2)')
plt.ylabel('Fraction positive')
plt.xlim(-1,8)
#plt.ylim(-0.05,0.8)
plt.show()

In [ ]:
noise_df[((noise_df.variant_id == 'allele')&(noise_df.WT_NLR>1)&(noise_df.WT_qY2H > 10)&(noise_df.qY2H == 0))]

In [ ]:
noise_df.variant_id.value_counts()

In [ ]:
noise_df[(noise_df.variant_id ==  'allele')&(noise_df.qY2H == 0)&(noise_df.WT_qY2H > 4)]

In [ ]:
df = noise_df[(noise_df.variant_id.isin(['wt','allele']))&(noise_df.WT_NLD > 1000)&(noise_df.db_symbol == 'MLH1')].copy()
df.sort_values(by='aa_pos',inplace=True)

# create figure and axis objects with subplots()
fig,ax = plt.subplots()
fig.set_size_inches(16, 9)
# make a plot
ax.plot(df.aa_change, df.NLR1.apply(lambda x: np.log2(x) if x >= 1 else 0), color="red", marker="o",linestyle='',label='NLR1',alpha=0.5)
ax.plot(df.aa_change, df.NLR2.apply(lambda x: np.log2(x) if x >= 1 else 0), color="orange", marker="o",linestyle='',label='NLR2',alpha=0.5)

# set x-axis label
ax.set_xlabel("aa_change")#,fontsize=14)
ax.set_xticklabels(labels=df.aa_change, rotation=45, ha="right")
# set y-axis label
ax.set_ylabel("NLR (log2)",color="red")#,fontsize=14)

plt.axhline(y=np.log2(noise_df[noise_df.variant_id == 'RRS'].NLR.max()), color='lightgrey', linestyle='--', label='max RRS')
plt.legend(loc='upper left')

# twin object for two different y-axis on the sample plot

ax2=ax.twinx()
# make a plot with different y-axis using second axis object
ax2.plot(df.aa_change, df["qY2H"],color="blue",marker="o",linestyle='',label='qY2H',alpha=0.5)
ax2.set_ylabel("qY2H",color="blue")#,fontsize=14)


plt.legend()
plt.show()

In [ ]:
df = noise_df[(noise_df.variant_id.isin(['wt','allele']))&(noise_df.WT_NLD > 1000)&(noise_df.db_symbol == 'MLH1')].copy()
df.sort_values(by='aa_pos',inplace=True)

# create figure and axis objects with subplots()
fig,ax = plt.subplots()
fig.set_size_inches(16, 9)
# make a plot
ax.plot(df.aa_change, df.NLD1.apply(lambda x: np.log10(x) if x >= 1 else 0), color="red", marker="o",linestyle='',label='NLD1',alpha=0.5)
ax.plot(df.aa_change, df.NLD2.apply(lambda x: np.log10(x) if x >= 1 else 0), color="orange", marker="o",linestyle='',label='NLD2',alpha=0.5)

# set x-axis label
ax.set_xlabel("aa_change")#,fontsize=14)
ax.set_xticklabels(labels=df.aa_change, rotation=45, ha="right")
# set y-axis label
ax.set_ylabel("NLD (log10)",color="red")#,fontsize=14)

plt.axhline(y=np.log10(noise_df[noise_df.variant_id == 'RRS'].NLD.max()), color='lightgrey', linestyle='--', label='max RRS')
plt.legend(loc='upper left')

# twin object for two different y-axis on the sample plot

ax2=ax.twinx()
# make a plot with different y-axis using second axis object
ax2.plot(df.aa_change, df["qY2H"],color="blue",marker="o",linestyle='',label='qY2H',alpha=0.5)
ax2.set_ylabel("qY2H",color="blue")#,fontsize=14)


plt.legend()
plt.show()

In [ ]:
len(noise_df[(noise_df.variant_id == 'wt')&(noise_df.NLD < 0)])

In [ ]:
len(noise_df[(noise_df.variant_id == 'wt')&(noise_df.NLD > 41)])

In [ ]:
len(noise_df[(noise_df.variant_id == 'wt')&(noise_df.NLD > 1000)])

In [ ]:
df = noise_df[(noise_df.variant_id == 'allele')&(noise_df.WT_NLR > 3)&(noise_df.db_symbol == 'MLH1')&(noise_df.qY2H > 0)]

data1 = df.NLR
data2 = df.qY2H

pearson_corr = pearsonr(data1, data2)
spearman_corr = spearmanr(data1, data2)
print('pearson',pearson_corr)
print('spearman',spearman_corr)
plt.figure(figsize=(5,5))
sns.scatterplot(x='qY2H', y='NLR', data=df);

In [ ]:
# df = noise_df[(noise_df.variant_id == 'allele')&(noise_df.WT_NLD > 1000)&(noise_df.db_symbol == 'MLH1')&(noise_df.qY2H > 0)]

# data1 = df.NLD
# data2 = df.qY2H

# pearson_corr = pearsonr(data1, data2)
# spearman_corr = spearmanr(data1, data2)
# print('pearson',pearson_corr)
# print('spearman',spearman_corr)
# plt.figure(figsize=(5,5))
# sns.scatterplot(x='qY2H', y='NLD', data=df);

In [ ]:
# df = noise_df[(noise_df.variant_id == 'allele')&(noise_df.WT_NLD > 1000)&(noise_df.db_symbol == 'MLH1')]

# data1 = df.NLD
# data2 = df.qY2H

# pearson_corr = pearsonr(data1, data2)
# spearman_corr = spearmanr(data1, data2)
# print('pearson',pearson_corr)
# print('spearman',spearman_corr)
# plt.figure(figsize=(5,5))
# sns.scatterplot(x='qY2H', y='NLD', data=df);

In [ ]:
df = noise_df[(noise_df.variant_id == 'allele')&(noise_df.WT_NLD > 1000)&(~df.NLD.isnull())&(~df.qY2H.isnull())]

data1 = df.NLD
data2 = df.qY2H

pearson_corr = pearsonr(data1, data2)
spearman_corr = spearmanr(data1, data2)
print('pearson',pearson_corr)
print('spearman',spearman_corr)
plt.figure(figsize=(5,5))
sns.scatterplot(x='qY2H', y='NLD', data=df);




In [ ]:
df = noise_df[(noise_df.variant_id == 'allele')&(noise_df.WT_NLD > 10000)&(~df.NLD.isnull())&(~df.qY2H.isnull())]

data1 = df.NLD
data2 = df.qY2H

pearson_corr = pearsonr(data1, data2)
spearman_corr = spearmanr(data1, data2)
print('pearson',pearson_corr)
print('spearman',spearman_corr)
plt.figure(figsize=(5,5))
sns.scatterplot(x='qY2H', y='NLD', data=df);





In [ ]:
noise_df[noise_df.variant_id == 'RRS'].NLD.max()

In [ ]:
noise_df[noise_df.variant_id == 'RRS'].NLD1.max()

In [ ]:
noise_df[noise_df.variant_id == 'RRS'].NLD2.max()

In [ ]:
noise_df[noise_df.variant_id == 'RRS'].NLR1.max()

In [ ]:
noise_df[noise_df.variant_id == 'RRS'].NLR2.max()

In [ ]:
noise_df[noise_df.variant_id == 'RRS'].NLR.max()

In [ ]:
noise_df[(noise_df.variant_id == 'allele')&(noise_df.NLD > 200)]

## Define significant perturbation threshold

In [ ]:
noise_df.head()